In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import fiona
from tqdm.auto import tqdm


# --------------------------
# 1. Read all layers
# --------------------------

layers = fiona.listlayers(gpkg_path)
gdfs = []
for layer in tqdm(layers):
    gdf = gpd.read_file(gpkg_path, layer=layer)
    gdfs.append(gdf)

# Merge all layers into one GeoDataFrame
gdf_all = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

# --------------------------
# 2. Define raster resolution + transform
# --------------------------
# Choose your desired resolution (in CRS units)
resolution = 10  # e.g., 10m grid

minx, miny, maxx, maxy = gdf_all.total_bounds
width = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)

transform = from_bounds(minx, miny, maxx, maxy, width, height)

# --------------------------
# 3. Rasterize — binary output
# --------------------------
shapes = ((geom, 1) for geom in tqdm(gdf_all.geometry))

raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,          # background
    dtype="uint8"
)

# --------------------------
# 4. Write to GeoTIFF
# --------------------------
with rasterio.open(
    out_tif, "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    transform=transform,
    crs=gdf_all.crs
) as dst:
    dst.write(raster, 1)

print("Raster created:", out_tif)


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/1273533 [00:00<?, ?it/s]

c:\Users\coach\myfiles\postdoc2\code\.venv\Lib\site-packages\rasterio\features.py:336: ShapeSkipWarning: Invalid or empty shape None at index 1273519 will not be rasterized.
  warnings.warn(
c:\Users\coach\myfiles\postdoc2\code\.venv\Lib\site-packages\rasterio\features.py:336: ShapeSkipWarning: Invalid or empty shape None at index 1273520 will not be rasterized.
  warnings.warn(
c:\Users\coach\myfiles\postdoc2\code\.venv\Lib\site-packages\rasterio\features.py:336: ShapeSkipWarning: Invalid or empty shape None at index 1273521 will not be rasterized.
  warnings.warn(
c:\Users\coach\myfiles\postdoc2\code\.venv\Lib\site-packages\rasterio\features.py:336: ShapeSkipWarning: Invalid or empty shape None at index 1273522 will not be rasterized.
  warnings.warn(
c:\Users\coach\myfiles\postdoc2\code\.venv\Lib\site-packages\rasterio\features.py:336: ShapeSkipWarning: Invalid or empty shape None at index 1273523 will not be rasterized.
  warnings.warn(
c:\Users\coach\myfiles\postdoc2\code\.venv\Li

Raster created: C:\Users\coach\myfiles\postdoc2\data\NIAPS\IAP_binary.tif


In [7]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from tqdm import tqdm

# ---------------- USER SETTINGS ----------------
gpkg_path = r"C:\Users\coach\myfiles\postdoc2\data\2023 NIAPS GeoPackage.gpkg"
out_tif = r"C:\Users\coach\myfiles\postdoc2\data\NIAPS\IAP_binary.tif"

RES = 10
TARGET_CRS = 32735
# ------------------------------------------------

print("Loading all layers from GeoPackage...")
layers = gpd.list_layers(gpkg_path)
print(f"Found {len(layers)} layers")

# Read and concatenate all polygon layers
gdf_list = []
for layer_name in layers['name']:
    if layer_name != 'layer_styles':
        try:
            gdf_temp = gpd.read_file(gpkg_path, layer=layer_name)
            gdf_list.append(gdf_temp)
            print(f"  Loaded: {layer_name} ({len(gdf_temp)} features)")
        except Exception as e:
            print(f"  Skipped: {layer_name}")

gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))
print(f"\nTotal features: {len(gdf)}")

# Reproject if needed
if gdf.crs.is_geographic:
    print("Reprojecting to target CRS...")
    gdf = gdf.to_crs(TARGET_CRS)

# Clean geometries
print("Cleaning geometries...")
gdf = gdf[~gdf.geometry.isna()]
gdf = gdf[~gdf.geometry.is_empty]
gdf['geometry'] = gdf.geometry.make_valid()

print(f"Valid features: {len(gdf)}")

# Union all geometries (this combines overlapping areas)
print("\nUnioning all polygons (this may take a while)...")
from shapely.ops import unary_union
unified_geom = unary_union(gdf.geometry.tolist())
print(f"Union complete. Result type: {unified_geom.geom_type}")

# Create a new GeoDataFrame with the unified geometry
if unified_geom.geom_type == 'MultiPolygon':
    unified_gdf = gpd.GeoDataFrame(
        {'geometry': list(unified_geom.geoms)},
        crs=gdf.crs
    )
else:
    unified_gdf = gpd.GeoDataFrame(
        {'geometry': [unified_geom]},
        crs=gdf.crs
    )

print(f"Unified to {len(unified_gdf)} polygon(s)")

# Get bounds
minx, miny, maxx, maxy = unified_gdf.total_bounds
width  = int(np.ceil((maxx - minx) / RES))
height = int(np.ceil((maxy - miny) / RES))
transform = from_bounds(minx, miny, maxx, maxy, width, height)

print(f"\nRaster dimensions: {width} x {height} pixels")
print(f"Estimated size: {(width * height) / (1024**3):.2f} GB")

# Create output directory
os.makedirs(os.path.dirname(out_tif), exist_ok=True)

# Prepare profile
profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "uint8",
    "crs": unified_gdf.crs,
    "transform": transform,
    "nodata": 0,
    "tiled": True,
    "compress": "LZW",
    "blockxsize": 256,
    "blockysize": 256,
    "BIGTIFF": "YES"  # Support for large files
}

print("\nRasterizing unified geometry...")

# Rasterize directly - only writes 1s where polygons exist
# This avoids creating a full array in memory
with rasterio.open(out_tif, "w", **profile) as dst:
    # Create generator of (geometry, value) pairs
    shapes = ((geom, 1) for geom in unified_gdf.geometry)
    
    # Rasterize in a memory-efficient way
    burned = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype='uint8',
        all_touched=False  # Only pixels with center inside polygon
    )
    
    dst.write(burned, 1)

print("\n✔ Finished: ", out_tif)
size_mb = os.path.getsize(out_tif) / (1024 * 1024)
print(f"Final size: {size_mb:.2f} MB")

Loading all layers from GeoPackage...
Found 15 layers
  Loaded: Acacia cyclops (39754 features)
  Loaded: Acacia saligna (30706 features)
  Loaded: Arundo donax (18203 features)
  Loaded: Chromolaena odorata (38072 features)
  Loaded: Eucalyptus_spp (270300 features)
  Loaded: Hakea spp (7717 features)
  Loaded: Lantana camara (55399 features)
  Loaded: Melia azedarach (62777 features)
  Loaded: Opuntia spp (71113 features)
  Loaded: Pinus_spp (95342 features)
  Loaded: Populus spp (70958 features)
  Loaded: Prosopis spp (68416 features)
  Loaded: Solanum mauritianum (69333 features)
  Loaded: Wattle_spp (375429 features)

Total features: 1273519
Reprojecting to target CRS...
Cleaning geometries...
Valid features: 1273519

Unioning all polygons (this may take a while)...
Union complete. Result type: MultiPolygon
Unified to 856094 polygon(s)

Raster dimensions: 161669 x 137579 pixels
Estimated size: 20.71 GB

Rasterizing unified geometry...


MemoryError: Unable to allocate 20.7 GiB for an array with shape (137579, 161669) and data type uint8

In [8]:
unified_geom

: 

: 

In [5]:
!uv pip install rasterio fiona

Resolved 11 packages in 353ms
Prepared 1 package in 8.59s
Installed 1 package in 220ms
 + fiona==1.10.1


In [3]:
!uv pip install scikit-learn tabpfn 

Resolved 55 packages in 2.57s
Prepared 34 packages in 51.35s
Installed 34 packages in 3.92s
 + annotated-types==0.7.0
 + anyio==4.12.0
 + backoff==2.2.1
 + distro==1.9.0
 + einops==0.8.1
 + eval-type-backport==0.3.1
 + filelock==3.20.0
 + fsspec==2025.12.0
 + h11==0.16.0
 + hf-xet==1.2.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.2.2
 + joblib==1.5.2
 + kditransform==1.2.0
 + llvmlite==0.46.0
 + mpmath==1.3.0
 + networkx==3.6.1
 + numba==0.63.1
 + posthog==6.9.3
 + pydantic==2.12.5
 + pydantic-core==2.41.5
 + pydantic-settings==2.12.0
 + python-dotenv==1.2.1
 + scikit-learn==1.7.2
 + scipy==1.16.3
 + shellingham==1.5.4
 + sympy==1.14.0
 + tabpfn==6.0.6
 + tabpfn-common-utils==0.2.10
 + threadpoolctl==3.6.0
 + torch==2.9.1
 + typer-slim==0.20.0
 + typing-inspection==0.4.2


In [8]:
import huggingface_hub
huggingface_hub.login(token = 'YOUR_HUGGINGFACE_TOKEN_HERE')

In [15]:
import torch
torch.cuda.is_available()
!nvidia-smi
print(torch.__version__)

Fri Dec 12 14:06:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1660 Ti   WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   60C    P8              4W /   80W |       1MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
!uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130 -U

Resolved 13 packages in 3.87s
Prepared 2 packages in 6m 12s
Uninstalled 2 packages in 3.93s
Installed 2 packages in 5.01s
 - numpy==2.3.4
 + numpy==2.3.5
 - torch==2.9.1
 + torch==2.9.1+cu130


In [2]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# Load example dataset
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Initialize and fit
model = TabPFNRegressor(device="auto")
# To use TabPFNv2:
# model = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
model.fit(X_train, y_train)

# Predict
preds = model.predict(X_test)

# Evaluate
print("MSE:", mean_squared_error(y_test, preds))
print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

MSE: 2616.539843486154
MAE: 40.77262732100813
R²: 0.5453654761808622


In [4]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# Load example dataset
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Initialize and fit
# model = TabPFNRegressor(device="auto")
# To use TabPFNv2:
model = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
model.fit(X_train, y_train)

# Predict
preds = model.predict(X_test)

# Evaluate
print("MSE:", mean_squared_error(y_test, preds))
print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

MSE: 2641.1836345195343
MAE: 40.82590050893287
R²: 0.5410835164662218
